In [ ]:
# Process Assessment Submission Notebook
# This notebook is triggered via HTTP after assessment submission
# It validates and stores the submitted assessment data

# Cell 1: Import libraries and setup
```python
import json
import pandas as pd
from datetime import datetime
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
```

# Cell 2: Accept and validate parameters
```python
# Parameters passed from HTTP trigger
assessmentToken = dbutils.widgets.get("assessmentToken", "")
assessmentCode = dbutils.widgets.get("assessmentCode", "")
responses_json = dbutils.widgets.get("responses", "[]")
submittedDate = dbutils.widgets.get("submittedDate", "")

logger.info(f"Processing submission: {assessmentToken}")

# Validate inputs
if not assessmentToken or not assessmentCode:
    raise ValueError("Missing required parameters: assessmentToken or assessmentCode")

try:
    responses = json.loads(responses_json)
except json.JSONDecodeError:
    raise ValueError("Invalid JSON in responses parameter")

logger.info(f"Received {len(responses)} responses")
```

# Cell 3: Transform response data
```python
# Convert responses to DataFrame format for storage
submission_records = []

for response in responses:
    record = {
        'AssessmentToken': assessmentToken,
        'AssessmentCode': assessmentCode,
        'QuestionCode': response.get('questionCode'),
        'ResponseValue': response.get('responseValue'),
        'ResponseType': type(response.get('responseValue')).__name__,
        'SubmittedDate': submittedDate,
        'ProcessedDate': datetime.now().isoformat(),
        'IsNumeric': isinstance(response.get('responseValue'), (int, float)),
    }
    submission_records.append(record)

responses_df = pd.DataFrame(submission_records)
logger.info(f"Transformed {len(responses_df)} response records")
print(responses_df.head())
```

# Cell 4: Load existing submissions table
```python
try:
    # Read existing submissions table from Lakehouse
    existing_df = spark.table("eprom_submissions").toPandas()
    logger.info(f"Loaded {len(existing_df)} existing submission records")
except Exception as e:
    logger.warning(f"Submissions table not found, creating new: {e}")
    existing_df = pd.DataFrame()
```

# Cell 5: Append new responses to table
```python
# Combine existing and new data
if not existing_df.empty:
    combined_df = pd.concat([existing_df, responses_df], ignore_index=True)
else:
    combined_df = responses_df

logger.info(f"Total records to write: {len(combined_df)}")

# Convert to Spark DataFrame
submissions_spark_df = spark.createDataFrame(combined_df)

# Write to Lakehouse table (overwrite mode)
submissions_spark_df.write.format("parquet").mode("overwrite").save(
    "abfss://Workspace@onelake.dfs.fabric.microsoft.com/LakedDatabase/eprom_submissions"
)

logger.info("✓ Successfully wrote submission data to eprom_submissions table")
```

# Cell 6: Create submission summary record
```python
# Create summary record for the assessment submission
summary_record = {
    'AssessmentToken': assessmentToken,
    'AssessmentCode': assessmentCode,
    'TotalResponses': len(responses),
    'NumericResponses': sum(1 for r in responses if isinstance(r.get('responseValue'), (int, float))),
    'TextResponses': sum(1 for r in responses if isinstance(r.get('responseValue'), str) and len(r.get('responseValue', '')) > 50),
    'SubmittedDate': submittedDate,
    'ProcessedDate': datetime.now().isoformat(),
    'ProcessingStatus': 'Completed',
    'ErrorMessage': None
}

summary_df = pd.DataFrame([summary_record])
print("\n=== Submission Summary ===")
print(summary_df.to_string(index=False))

# Store summary (optional - separate table for quick dashboard queries)
summary_spark_df = spark.createDataFrame(summary_df)
summary_spark_df.write.format("parquet").mode("append").save(
    "abfss://Workspace@onelake.dfs.fabric.microsoft.com/LakedDatabase/eprom_submission_summaries"
)

logger.info("✓ Submission processed and logged")
```

# Cell 7: Return status
```python
result = {
    "success": True,
    "assessmentToken": assessmentToken,
    "recordsProcessed": len(responses),
    "timestamp": datetime.now().isoformat()
}

print("\n=== PROCESSING COMPLETE ===")
print(json.dumps(result, indent=2))
```